In [1]:
!apt-get install openjdk-11-jdk-headless -qq
!pip install -q pyspark findspark

In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
# No need to download Spark—pip installation includes Spark JARs

import findspark
findspark.init()

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("ColabSpark").getOrCreate()
spark

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# --- 1. Paths and input ---
DATA_PATH = "/content/drive/MyDrive/Infosys722/samadult.csv"   # NHIS 2018 Sample Adult CSV

# Load CSV with inferred schema; treat empty string/NA as null
df_raw = spark.read.csv(DATA_PATH, header=True, inferSchema=True, nullValue="", nanValue="NA")
print("Raw shape (rows, cols):", df_raw.count(), len(df_raw.columns))

Raw shape (rows, cols): 25417 742


In [5]:
from pathlib import Path
OUT_DIR = Path("/content/drive/MyDrive/Infosys722/iteration4_prep_outputs")

In [6]:
# ✅ ensure directory exists
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [7]:
# --- 2. Target and BAN variables ---
TARGET = "AMIGR"   # target column (not used as feature)
design_id_all = ["HHX","FMX","FPX","WTIA_SA","WTFA_SA","PSTRAT","PPSU","SRVY_YR","RECTYPE","INTV_QRT","INTV_MON"]
DESIGN_ID = [c for c in design_id_all if c in df_raw.columns]
BAN = set(DESIGN_ID + [TARGET])
print("BAN variables:", sorted(BAN))

BAN variables: ['AMIGR', 'FMX', 'FPX', 'HHX', 'INTV_MON', 'INTV_QRT', 'PPSU', 'PSTRAT', 'RECTYPE', 'SRVY_YR', 'WTFA_SA', 'WTIA_SA']


In [8]:
# --- 3. Variable quality: constant columns & variable-specific invalid codes ---
from pyspark.sql import functions as F

# ============================================================
# 3.0 Default invalid codes (used when variable not in map)
# ============================================================
DEFAULT_INVALID = {7, 8, 9, 97, 98, 99, 997, 998, 999, 9999}

# ============================================================
# 3.1 Variable-specific invalid codes
# ============================================================
# Derived from NHIS 2018 Sample Adult Layout (samadult_layout.pdf)
invalid_code_map = {
    # Demographic & socio-economic
    "SEX":        {7, 8, 9},       # valid 1/2 only
    "REGION":     set(),           # valid 1..4
    "HISPAN_I":   {10, 11},        # 10=Refused, 11=Not ascertained
    "R_MARITL":   {7, 8, 9},       # refused/not ascertained/unknown
    "DOINGLWA":   {7, 8, 9},

    # Health condition: ever told (1/2 valid, 7/8/9 invalid)
    "HYPEV":      {7, 8, 9},
    "CHDEV":      {7, 8, 9},
    "MIEV":       {7, 8, 9},
    "HRTEV":      {7, 8, 9},
    "CANEV":      {7, 8, 9},
    "HVPEV":      {7, 8, 9},
    "ASPONOWN":   {7, 8, 9},

    # Lifestyle / habits
    "SMKSTAT2":   {7, 8, 9},
    "SMKNOW":     {7, 8, 9},
    "ALC1YR":     {7, 8, 9},

    # Pain / allergy
    "AHAYFYR":    {7, 8, 9},
    "PAINFACE":   {7, 8, 9},
    "PAINLB":     {7, 8, 9},
    "PAIN_2A":    {7, 8, 9},

    # Mental health / tiredness
    "DEP_1":      {7, 8, 9},
    "ANX_1":      {7, 8, 9},
    "TIRED_1":    {7, 8, 9},

    # Diabetes / chronic conditions
    "DIBPRE2":    {7, 8, 9},

    # Work / worry
    "AWORPAY":    {7, 8, 9},

    # Medication adherence (Rx)
    "APX12_1":    {7, 8, 9},
    "ARX12_2":    {7, 8, 9},
    "ARX12_3":    {7, 8, 9},

    # Duration-type or day-count variables
    "BEDDAYR":    {997, 998, 999},
    "WKDAYR":     {997, 998, 999},
    "ALC12MYR":   {997, 998, 999},
    "ALC12MTP":   {7, 8, 9},
    "YRSWRKPA":   {97, 98, 99},

    # Numeric variables (handle with range rules)
    "BMI":        {9999},
}

# ============================================================
# 3.2 Numeric range rules (layout-based validity)
# ============================================================
numeric_range_rules = {
    "AGE_P":     (18, 85),     # valid 18–85+
    "ASISLEEP":  (1, 24),      # 1–24 hours per day
    "BMI":       (1000, 9995), # valid coded range
    "BEDDAYR":   (0, 366),
    "WKDAYR":    (0, 366),
    "ALC12MYR":  (0, 366),
    "YRSWRKPA":  (0, 35)
}

# ============================================================
# 3.3 Count distinct non-null values (for constant columns)
# ============================================================
def count_distinct_non_null(col):
    return F.countDistinct(F.when(~F.col(col).isNull(), F.col(col)))

nunique_df = []
for c in df_raw.columns:
    nunq = df_raw.select(count_distinct_non_null(c).alias("nunq")).collect()[0]["nunq"]
    nunique_df.append((c, nunq))

nunique_sdf = spark.createDataFrame(nunique_df, ["col", "nunique_non_null"])
constant_vars_raw = [r["col"] for r in nunique_sdf.filter(F.col("nunique_non_null") <= 1).collect()]

# ============================================================
# 3.4 Compute effective missing ratio per variable
# ============================================================
def effective_missing_ratio(df, col):
    c = F.col(col)
    as_num = c.cast("double")

    # ① Nulls
    null_cond = F.when(c.isNull(), 1).otherwise(0)

    # ② Invalid codes (variable-specific or fallback)
    codes = invalid_code_map.get(col, DEFAULT_INVALID)
    invalid_cond = F.when(as_num.isin(list(codes)), 1).otherwise(0)

    # ③ Out-of-range values (if applicable)
    if col in numeric_range_rules:
        lo, hi = numeric_range_rules[col]
        range_cond = F.when((as_num < lo) | (as_num > hi), 1).otherwise(0)
    else:
        range_cond = F.lit(0)

    # ④ Combine all
    is_missing = F.when((null_cond + invalid_cond + range_cond) > 0, 1).otherwise(0)

    # ⑤ Compute ratio
    mr = df.select(F.mean(is_missing.cast("double")).alias("mr")).collect()[0]["mr"]
    return float(mr or 0.0)

eff_miss_list = [(c, effective_missing_ratio(df_raw, c)) for c in df_raw.columns]
eff_miss_sdf = spark.createDataFrame(eff_miss_list, ["col", "eff_missing"])

# ============================================================
# 3.5 Combine results for summary table
# ============================================================
quality_summary = (
    nunique_sdf.join(eff_miss_sdf, on="col", how="inner")
                .orderBy(F.desc("eff_missing"))
)

# Show top 50 variables by missingness
quality_summary.show(50, truncate=False)

print(f"\n✅ Constant variables detected: {len(constant_vars_raw)}")

+--------+----------------+------------------+
|col     |nunique_non_null|eff_missing       |
+--------+----------------+------------------+
|ALDURA91|2               |0.9999606562536885|
|ALTIME91|2               |0.9999606562536885|
|ALDURA29|3               |0.9999213125073769|
|ALTIME29|3               |0.9999213125073769|
|ALDURB91|1               |0.9999213125073769|
|ALCHRC91|1               |0.9999213125073769|
|ALUNIT91|1               |0.9999213125073769|
|ALDURB29|1               |0.9998819687610654|
|ALCHRC29|1               |0.9998819687610654|
|ALUNIT29|1               |0.9998819687610654|
|CANAGE9 |5               |0.9998032812684424|
|ALTIME27|7               |0.9996459062831963|
|CANAGE11|8               |0.9996459062831963|
|ALTIME34|11              |0.9996459062831963|
|ALCHRC27|3               |0.9996065625368847|
|ALDURB27|4               |0.9996065625368847|
|ALDURA27|9               |0.9996065625368847|
|SHINGWHN|3               |0.9996065625368847|
|ALUNIT27|4  

In [9]:
# Export a full quality report (for audit trail)
quality_summary = nunique_sdf.join(eff_miss_sdf, on="col", how="inner")
(quality_summary
 .orderBy(F.desc("eff_missing"))
 .toPandas()
 .to_csv(OUT_DIR/"quality_summary_all_vars.csv", index=False))
print("Saved variable quality audit to:", OUT_DIR/"quality_summary_all_vars.csv")

Saved variable quality audit to: /content/drive/MyDrive/Infosys722/iteration4_prep_outputs/quality_summary_all_vars.csv


In [10]:
# ===============================================================
# 4. Build variable pools (domain_vars + HQ pool)
# ===============================================================

# 4.1 Domain knowledge variables (manual curated list)
domain_vars_seed = [
    "ASISLEEP","BMI","SMKSTAT2","DOINGLWA","HYPEV","CHDEV","MIEV","HRTEV",
    "CANEV","AHAYFYR","PAINFACE","PAINLB","DEP_1","ANX_1","TIRED_1",
    "SEX","AGE_P","REGION","HISPAN_I","RACERPI2","R_MARITL","HVPEV",
    "ASPONOWN","DIBPRE2","SMKNOW","AWORPAY","APX12_1","ARX12_2","ARX12_3",
    "ASIMEDC","ASIMUCH","PAIN_2A"
]

# Keep only variables that exist in dataset and not banned
domain_vars = [c for c in domain_vars_seed if c in df_raw.columns and c not in BAN]

# 4.2 High-quality variable pool (non-constant, not banned, missing ≤ threshold)
MISSING_TH = 0.40
eff_miss_map = {r["col"]: r["eff_missing"] for r in eff_miss_sdf.collect()}

hq_pool = [c for c in df_raw.columns
           if c not in BAN
           and c not in set(constant_vars_raw)
           and eff_miss_map.get(c, 1.0) <= MISSING_TH]

print(f"domain_vars: {len(domain_vars)} | hq_pool: {len(hq_pool)} | constants dropped: {len(constant_vars_raw)}")

# Save both lists for reproducibility
OUT_DIR_PATH = Path(OUT_DIR)
OUT_DIR_PATH.mkdir(parents=True, exist_ok=True)

with open(OUT_DIR_PATH/"domain_vars.txt", "w") as f:
    f.write("\n".join(domain_vars))
with open(OUT_DIR_PATH/"hq_pool_vars.txt", "w") as f:
    f.write("\n".join(hq_pool))

print("Saved variable lists: domain_vars.txt and hq_pool_vars.txt")

domain_vars: 30 | hq_pool: 225 | constants dropped: 24
Saved variable lists: domain_vars.txt and hq_pool_vars.txt


In [11]:
# ===============================================================
# Step 5 — Feature Selection with Random Forest (PySpark version)
# ===============================================================
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Pipeline
from pathlib import Path
import pandas as pd

# ---------------------------------------------------------------
# Build modelling table (target + HQ pool)
# ---------------------------------------------------------------
# Keep only target + HQ pool variables
df_use = df_raw.select([TARGET] + hq_pool)

# Keep only valid target codes {1,2}
df_use = df_use.filter(F.col(TARGET).isin([1, 2]))

# Encode target as 0/1  (1 → 1, 2 → 0)
df_use = df_use.withColumn("__label__", (F.col(TARGET) == 1).cast("double"))

# Drop the original target column
df_use = df_use.drop(TARGET)

# Cast all predictors to double for Spark ML
for c, t in df_use.dtypes:
    if c != "__label__":
        df_use = df_use.withColumn(c, F.col(c).cast("double"))

# Replace NaN / Null / ±Inf with 0 (consistent with Iteration 3)
for c in df_use.columns:
    if c != "__label__":
        df_use = df_use.withColumn(
            c,
            F.when(
                (F.isnan(F.col(c))) |
                (F.col(c).isNull()) |
                (F.col(c) == float('inf')) |
                (F.col(c) == float('-inf')),
                F.lit(0)
            ).otherwise(F.col(c))
        )

# Drop constant variables (no variance)
valid_cols = []
for c in df_use.columns:
    if c == "__label__":
        continue
    nunq = df_use.select(F.countDistinct(F.col(c))).collect()[0][0]
    if nunq > 1:
        valid_cols.append(c)

print(f"✅ Valid feature count: {len(valid_cols)} / {len(hq_pool)}")

# ---------------------------------------------------------------
# 5) Random Forest pipeline
# ---------------------------------------------------------------
assembler = VectorAssembler(inputCols=valid_cols, outputCol="features")

rf = RandomForestClassifier(
    labelCol="__label__",
    featuresCol="features",
    numTrees=200,
    maxDepth=30,
    minInstancesPerNode=200,   # ≈ min_samples_split
    minWeightFractionPerNode=0.0,
    seed=42,
    impurity="gini"
)

pipeline = Pipeline(stages=[assembler, rf])

# Train-test split (80/20 stratified by random seed)
train_df, test_df = df_use.randomSplit([0.8, 0.2], seed=42)
print(f"Training rows: {train_df.count()} | Testing rows: {test_df.count()}")

# Fit model
model = pipeline.fit(train_df)
print("✅ Random Forest model fitted successfully!")

# Predict
preds = model.transform(test_df)

# ---------------------------------------------------------------
# 5.1 Evaluate model performance
# ---------------------------------------------------------------
evaluator = BinaryClassificationEvaluator(labelCol="__label__", rawPredictionCol="rawPrediction")
auc = evaluator.evaluate(preds, {evaluator.metricName: "areaUnderROC"})
print(f"✅ ROC-AUC (test): {auc:.3f}")

# ---------------------------------------------------------------
# 5.2 Extract and save feature importances
# ---------------------------------------------------------------
rf_model = model.stages[-1]
importances = rf_model.featureImportances.toArray()

imp_df = pd.DataFrame({
    "feature": valid_cols,
    "importance": importances[:len(valid_cols)]
}).sort_values("importance", ascending=False)

# Aggregate by feature (1-to-1 mapping here)
imp_agg = imp_df.groupby("feature", as_index=False)["importance"].sum() \
                .sort_values("importance", ascending=False)

# Save full importances
OUT_DIR_PATH = Path(OUT_DIR)
OUT_DIR_PATH.mkdir(parents=True, exist_ok=True)
imp_agg.to_csv(OUT_DIR_PATH / "rf_feature_importance_aggregated.csv", index=False)
print("📄 Saved → rf_feature_importance_aggregated.csv")

# ---------------------------------------------------------------
# 5.3 Select top-N important variables
# ---------------------------------------------------------------
TOP_N = 40
top_vars_data_driven = imp_agg.head(TOP_N)["feature"].tolist()
print(f"✅ Top {TOP_N} features selected:\n{top_vars_data_driven}")

✅ Valid feature count: 225 / 225
Training rows: 20433 | Testing rows: 4970
✅ Random Forest model fitted successfully!
✅ ROC-AUC (test): 0.784
📄 Saved → rf_feature_importance_aggregated.csv
✅ Top 40 features selected:
['PAINECK', 'PAINFACE', 'PAIN_4', 'BEDDAYR', 'ASIREST', 'PAINLB', 'TIRED_2', 'ANX_1', 'ASIEFFRT', 'DEP_1', 'ASISLPFL', 'TIRED_1', 'ASIHOPLS', 'ASIRSTLS', 'ASINERV', 'AGE_P', 'PAIN_2A', 'ANX_3R', 'ASISAD', 'ASIWTHLS', 'AHCAFYR1', 'TIRED_3', 'SHTHPV2', 'FLSIT', 'FLSOCL', 'ANX_2', 'ASISLPST', 'ASIRETR', 'ASIMEDC', 'COG_SS', 'AHERNOY2', 'YRSWRKPA', 'ASINBILL', 'ASISTLV', 'DEP_2', 'AHCNOYR2', 'AHCAFYR3', 'FLA1AR', 'ASIHIVT', 'ASISLPMD']


In [12]:
# ===============================================================
# 6) Extract and aggregate feature importances
# ===============================================================
import shutil

rf_model = model.stages[-1]  # The fitted Random Forest model
importances = rf_model.featureImportances.toArray()

# Map importances back to feature names
imp_df = pd.DataFrame({
    "feature": valid_cols,
    "importance": importances[:len(valid_cols)]
}).sort_values("importance", ascending=False)

# Aggregate importances (already one-to-one since no OHE here)
imp_agg = imp_df.groupby("feature", as_index=False)["importance"].sum() \
                .sort_values("importance", ascending=False)

# Save full list for transparency
OUT_DIR_PATH = Path(OUT_DIR)
OUT_DIR_PATH.mkdir(parents=True, exist_ok=True)
imp_agg.to_csv(OUT_DIR_PATH / "rf_feature_importance_aggregated.csv", index=False)
print("📄 Saved feature importances → rf_feature_importance_aggregated.csv")

# Select top-N features for data-driven variable set
TOP_N = 40
top_vars_data_driven = imp_agg.head(TOP_N)["feature"].tolist()
print(f"✅ Top {TOP_N} important variables selected by Random Forest:")
print(top_vars_data_driven)

# ===============================================================
# 7) Final variable union (domain + data-driven) and export
# ===============================================================
# Combine data-driven features and manually curated domain variables
final_vars = sorted((set(domain_vars) | set(top_vars_data_driven)) - set(BAN))
print(f"✅ Final variable count: {len(final_vars)}")

# Select only target + final variables for next step
df_final = df_raw.select([TARGET] + final_vars)

# Write single CSV file (matching Iteration 3 convention)
(df_final
 .coalesce(1)
 .write
 .mode("overwrite")
 .option("header", True)
 .csv(str(OUT_DIR_PATH / "final_selected_table_tmp")))

# Rename part file to consistent name
tmp_dir = OUT_DIR_PATH / "final_selected_table_tmp"
part_file = [p for p in tmp_dir.glob("part-*.csv")][0]
out_csv = OUT_DIR_PATH / "final_selected_table.csv"
shutil.move(str(part_file), str(out_csv))
shutil.rmtree(tmp_dir)

print(f"✅ Saved final selected table → {out_csv}")

📄 Saved feature importances → rf_feature_importance_aggregated.csv
✅ Top 40 important variables selected by Random Forest:
['PAINECK', 'PAINFACE', 'PAIN_4', 'BEDDAYR', 'ASIREST', 'PAINLB', 'TIRED_2', 'ANX_1', 'ASIEFFRT', 'DEP_1', 'ASISLPFL', 'TIRED_1', 'ASIHOPLS', 'ASIRSTLS', 'ASINERV', 'AGE_P', 'PAIN_2A', 'ANX_3R', 'ASISAD', 'ASIWTHLS', 'AHCAFYR1', 'TIRED_3', 'SHTHPV2', 'FLSIT', 'FLSOCL', 'ANX_2', 'ASISLPST', 'ASIRETR', 'ASIMEDC', 'COG_SS', 'AHERNOY2', 'YRSWRKPA', 'ASINBILL', 'ASISTLV', 'DEP_2', 'AHCNOYR2', 'AHCAFYR3', 'FLA1AR', 'ASIHIVT', 'ASISLPMD']
✅ Final variable count: 62
✅ Saved final selected table → /content/drive/MyDrive/Infosys722/iteration4_prep_outputs/final_selected_table.csv
